# 📈 EMA Crossover Trading Bot — MT5 + Telegram

**Full pipeline:** Data → Strategy → Backtesting → Performance Metrics → Risk Management → Live Execution → Telegram Notifications

---

| Section | Description |
|---|---|
| 1 | Installation & Imports |
| 2 | Configuration |
| 3 | MT5 Connection |
| 4 | Data Fetching |
| 5 | EMA Strategy Engine |
| 6 | Backtesting |
| 7 | Performance Metrics (Sharpe, Sortino, Calmar) |
| 8 | Visualisation |
| 9 | Parameter Optimisation |
| 10 | Risk Management (SL/TP, Position Sizing, Trailing Stop) |
| 11 | Live Signal Generation |
| 12 | Order Execution |
| 13 | Telegram Integration |
| 14 | Bot Loop & Scheduler |
| 15 | Launch the Bot |

> ⚠️ **Always test on a demo account first before going live.**

## 1. 📦 Installation & Imports

In [ ]:
# Run once to install all dependencies
!pip install MetaTrader5 pandas numpy matplotlib python-telegram-bot schedule -q

In [ ]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from itertools import product
from datetime import datetime, timedelta
import time
import schedule
import asyncio
import threading
import logging
import warnings
warnings.filterwarnings('ignore')

# Telegram
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, CommandHandler, CallbackQueryHandler,
    ContextTypes
)

# Plot styling
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'text.color':       '#c9d1d9',
})

print('✅ All imports successful.')

## 2. ⚙️ Configuration

> **Edit this cell** with your credentials and trading preferences before running anything else.

In [ ]:
CONFIG = {
    # ── MT5 Credentials ───────────────────────────────────────────
    'MT5_LOGIN':    12345678,          # ← Your MT5 account number
    'MT5_PASSWORD': 'your_password',   # ← MT5 password
    'MT5_SERVER':   'Broker-Server',   # ← e.g. 'ICMarkets-Demo'

    # ── Trading Parameters ─────────────────────────────────────────
    'SYMBOL':       'XAUUSD',          # Instrument: Gold, EURUSD, GBPUSD …
    'TIMEFRAME':    mt5.TIMEFRAME_H1,  # H1 | H4 | D1 | M15 | M30
    'EMA_SHORT':    9,                 # Short EMA period
    'EMA_LONG':     21,                # Long EMA period
    'LOT_SIZE':     0.01,              # Default lot size
    'MAGIC':        202400,            # Unique bot order ID

    # ── Risk Management ────────────────────────────────────────────
    'SL_PIPS':           50,           # Stop-loss distance in pips
    'TP_PIPS':           100,          # Take-profit distance in pips (2:1 R:R)
    'TRAILING_SL':       True,         # Enable trailing stop-loss
    'TRAIL_PIPS':        30,           # Trailing distance in pips
    'MAX_DRAWDOWN_PCT':  5.0,          # Halt bot if drawdown exceeds this %
    'RISK_PER_TRADE_PCT': 1.0,         # Risk this % of balance per trade

    # ── Telegram ──────────────────────────────────────────────────
    'TG_TOKEN':   'YOUR_BOT_TOKEN',    # ← From @BotFather
    'TG_CHAT_ID': 'YOUR_CHAT_ID',      # ← From @userinfobot

    # ── Bot Control ────────────────────────────────────────────────
    'CHECK_INTERVAL_SEC': 60,          # Scan frequency in seconds
    'BACKTEST_BARS':      500,         # Bars used for backtesting
    'LOG_FILE':           'bot.log',
}

print('CONFIG loaded:')
for k, v in CONFIG.items():
    if 'PASSWORD' in k or 'TOKEN' in k:
        print(f'  {k:<25} *** (hidden)')
    else:
        print(f'  {k:<25} {v}')

## 3. 🔌 MT5 Connection

In [ ]:
# ── Logging setup ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(CONFIG['LOG_FILE']),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)


def connect_mt5() -> bool:
    """Initialize and log in to MT5."""
    if not mt5.initialize():
        log.error(f'MT5 initialize failed: {mt5.last_error()}')
        return False
    authorized = mt5.login(
        CONFIG['MT5_LOGIN'],
        password=CONFIG['MT5_PASSWORD'],
        server=CONFIG['MT5_SERVER']
    )
    if not authorized:
        log.error(f'MT5 login failed: {mt5.last_error()}')
        mt5.shutdown()
        return False
    log.info('✅ MT5 connected successfully.')
    return True


def disconnect_mt5():
    """Cleanly shut down MT5 connection."""
    mt5.shutdown()
    log.info('MT5 disconnected.')


def get_account_info() -> dict:
    """Return account balance, equity, margin level."""
    info = mt5.account_info()
    if info is None:
        return {}
    return {
        'balance':      info.balance,
        'equity':       info.equity,
        'margin':       info.margin,
        'free_margin':  info.margin_free,
        'margin_level': info.margin_level,
        'profit':       info.profit,
        'currency':     info.currency,
        'leverage':     info.leverage,
        'name':         info.name,
    }


# ── Connect and print account summary ─────────────────────────────
if connect_mt5():
    acct = get_account_info()
    print('\n📋 ACCOUNT SUMMARY')
    print('─' * 35)
    for k, v in acct.items():
        print(f'  {k:<18} {v}')
else:
    print('❌ MT5 connection failed. Check CONFIG credentials.')

## 4. 📊 Data Fetching

In [ ]:
def fetch_ohlcv(symbol: str, timeframe: int, bars: int) -> pd.DataFrame:
    """Fetch OHLCV bars from MT5 into a tidy DataFrame."""
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, bars)
    if rates is None or len(rates) == 0:
        log.error(f'No data for {symbol}')
        return pd.DataFrame()
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    df.rename(columns={'close': 'price'}, inplace=True)
    df['returns'] = np.log(df['price'] / df['price'].shift(1))
    return df


# ── Preview data ────────────────────────────────────────────────────
df_raw = fetch_ohlcv(CONFIG['SYMBOL'], CONFIG['TIMEFRAME'], CONFIG['BACKTEST_BARS'])

print(f'Fetched {len(df_raw)} bars for {CONFIG["SYMBOL"]}')
print(f'Date range: {df_raw.index[0]}  →  {df_raw.index[-1]}')
print()
df_raw[['open','high','low','price','tick_volume','returns']].tail(10)

## 5. 📐 EMA Strategy Engine

The `EMABacktester` class is the core engine. Key differences vs the original SMA version:

- Uses `.ewm(span=n, adjust=False).mean()` — true exponential weighting
- EMAs react faster to recent price moves, reducing lag on crossover signals
- Fully vectorised (no loops) for speed

In [ ]:
class EMABacktester:
    """
    Vectorized EMA crossover backtester.
    Fetches data directly from MT5 and computes full performance + risk metrics.
    """

    def __init__(self, symbol: str, ema_short: int, ema_long: int,
                 timeframe=None, bars=None):
        self.symbol    = symbol
        self.ema_short = ema_short
        self.ema_long  = ema_long
        self.timeframe = timeframe or CONFIG['TIMEFRAME']
        self.bars      = bars      or CONFIG['BACKTEST_BARS']
        self.results   = None
        self.results_overview = None
        self._load_data()

    def __repr__(self):
        return (f'EMABacktester(symbol={self.symbol}, '
                f'EMA {self.ema_short}/{self.ema_long}, '
                f'bars={self.bars})')

    # ── Data ───────────────────────────────────────────────────────
    def _load_data(self):
        df = fetch_ohlcv(self.symbol, self.timeframe, self.bars)
        if df.empty:
            raise RuntimeError('Failed to load data from MT5.')
        self.data = df
        self._prepare()

    def _prepare(self):
        d = self.data.copy()
        d['EMA_Short'] = d['price'].ewm(span=self.ema_short, adjust=False).mean()
        d['EMA_Long']  = d['price'].ewm(span=self.ema_long,  adjust=False).mean()
        self.data = d

    # ── Update parameters on-the-fly ───────────────────────────────
    def set_parameters(self, ema_short=None, ema_long=None):
        """Update EMA spans and recalculate indicators."""
        if ema_short is not None:
            self.ema_short = ema_short
            self.data['EMA_Short'] = self.data['price'].ewm(
                span=self.ema_short, adjust=False).mean()
        if ema_long is not None:
            self.ema_long = ema_long
            self.data['EMA_Long'] = self.data['price'].ewm(
                span=self.ema_long, adjust=False).mean()

    # ── Core backtest logic ────────────────────────────────────────
    def test_strategy(self) -> tuple:
        """
        Vectorized backtest:
          position  = +1 (long)  when EMA_Short > EMA_Long
          position  = -1 (short) when EMA_Short < EMA_Long
          strategy  = yesterday's position × today's log return
        Returns (strategy_perf, outperformance_vs_hold)
        """
        d = self.data.copy().dropna()
        d['position']  = np.where(d['EMA_Short'] > d['EMA_Long'], 1, -1)
        d['strategy']  = d['position'].shift(1) * d['returns']
        d.dropna(inplace=True)
        d['creturns']  = d['returns'].cumsum().apply(np.exp)   # buy-and-hold
        d['cstrategy'] = d['strategy'].cumsum().apply(np.exp)  # EMA strategy
        self.results = d

        perf    = d['cstrategy'].iloc[-1]
        outperf = perf - d['creturns'].iloc[-1]
        return round(perf, 6), round(outperf, 6)

print('✅ EMABacktester class defined.')

## 6. 🔁 Run Backtest

In [ ]:
# ── Initialise backtester ──────────────────────────────────────────
bt = EMABacktester(
    symbol    = CONFIG['SYMBOL'],
    ema_short = CONFIG['EMA_SHORT'],
    ema_long  = CONFIG['EMA_LONG'],
)

# ── Run ────────────────────────────────────────────────────────────
perf, outperf = bt.test_strategy()

print(f'Strategy Performance  : {perf:.4f}x  ({(perf-1)*100:.2f}%)')
print(f'Buy-and-Hold Return   : {bt.results["creturns"].iloc[-1]:.4f}x')
print(f'Outperformance        : {outperf:+.4f}x  ({outperf*100:+.2f}%)')
print()
print('Sample of results DataFrame:')
bt.results[['price','EMA_Short','EMA_Long','position','strategy','creturns','cstrategy']].tail(8)

## 7. 📏 Performance Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **Sharpe** | (Rp − Rf) / σ × √252 | Return per unit of total risk |
| **Sortino** | (Rp − Rf) / σ_down × √252 | Like Sharpe but only penalises downside vol |
| **Calmar** | CAGR / \|Max Drawdown\| | Return per unit of max drawdown |
| **Max Drawdown** | min((V − peak) / peak) | Worst peak-to-trough loss |

In [ ]:
def compute_metrics(strategy_returns: pd.Series,
                    cumulative_returns: pd.Series,
                    risk_free_rate: float = 0.02) -> dict:
    """
    Compute a comprehensive set of performance and risk metrics.

    Parameters
    ----------
    strategy_returns  : pd.Series  — daily log returns of the strategy
    cumulative_returns: pd.Series  — cumulative multiplier (starts at 1.0)
    risk_free_rate    : float      — annual risk-free rate (default 2%)
    """
    strat    = strategy_returns.dropna()
    cum      = cumulative_returns.dropna()
    n_years  = len(strat) / 252

    # ── Return metrics ─────────────────────────────────────────────
    total_return = cum.iloc[-1] - 1
    cagr         = (cum.iloc[-1] ** (1 / max(n_years, 1e-6))) - 1

    # ── Sharpe Ratio ───────────────────────────────────────────────
    daily_rf = risk_free_rate / 252
    excess   = strat - daily_rf
    sharpe   = (excess.mean() / excess.std() * np.sqrt(252)
                if excess.std() != 0 else 0.0)

    # ── Sortino Ratio ──────────────────────────────────────────────
    downside     = strat[strat < 0]
    downside_std = (downside.std() * np.sqrt(252)
                    if len(downside) > 1 else 1e-9)
    sortino      = (strat.mean() * 252 - risk_free_rate) / downside_std

    # ── Max Drawdown ───────────────────────────────────────────────
    roll_max = cum.cummax()
    drawdown = (cum - roll_max) / roll_max
    max_dd   = drawdown.min()

    # ── Calmar Ratio ───────────────────────────────────────────────
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0.0

    # ── Trade-level stats ──────────────────────────────────────────
    trades        = strat[strat != 0]
    wins          = trades[trades > 0]
    losses        = trades[trades < 0]
    win_rate      = len(wins) / len(trades) * 100 if len(trades) > 0 else 0
    profit_factor = (wins.sum() / abs(losses.sum())
                     if len(losses) > 0 and losses.sum() != 0 else np.nan)

    # ── Volatility ─────────────────────────────────────────────────
    ann_vol = strat.std() * np.sqrt(252)

    return {
        'Total Return (%)':  round(total_return * 100, 2),
        'CAGR (%)':          round(cagr * 100, 2),
        'Ann. Volatility (%)': round(ann_vol * 100, 2),
        'Sharpe Ratio':      round(sharpe, 3),
        'Sortino Ratio':     round(sortino, 3),
        'Calmar Ratio':      round(calmar, 3),
        'Max Drawdown (%)':  round(max_dd * 100, 2),
        'Win Rate (%)':      round(win_rate, 2),
        'Profit Factor':     round(profit_factor, 3),
        'Total Trades':      len(trades),
        'Avg Trade (log)':   round(trades.mean(), 6) if len(trades) > 0 else 0,
        'Best Trade (log)':  round(trades.max(), 6)  if len(trades) > 0 else 0,
        'Worst Trade (log)': round(trades.min(), 6)  if len(trades) > 0 else 0,
    }


# ── Print metrics ──────────────────────────────────────────────────
metrics = compute_metrics(bt.results['strategy'], bt.results['cstrategy'])

print(f'\n📊 PERFORMANCE REPORT — {CONFIG["SYMBOL"]}  EMA {CONFIG["EMA_SHORT"]}/{CONFIG["EMA_LONG"]}')
print('═' * 45)
for k, v in metrics.items():
    bar = ''
    if k == 'Sharpe Ratio':
        bar = ' ⭐⭐⭐' if v >= 2 else (' ⭐⭐' if v >= 1 else ' ⭐')
    elif k == 'Max Drawdown (%)':
        bar = ' 🟢' if v > -10 else (' 🟡' if v > -20 else ' 🔴')
    print(f'  {k:<25}  {v}{bar}')

## 8. 📉 Visualisation

In [ ]:
def plot_strategy(bt: EMABacktester):
    """4-panel strategy dashboard."""
    res = bt.results
    fig, axes = plt.subplots(4, 1, figsize=(14, 18), sharex=True,
                             gridspec_kw={'height_ratios': [3, 1.5, 1.5, 1.5]})
    fig.suptitle(
        f'{bt.symbol}  —  EMA {bt.ema_short}/{bt.ema_long} Crossover',
        fontsize=16, fontweight='bold', color='#f0f6fc', y=0.98
    )

    # ── Panel 1: Price + EMAs ──────────────────────────────────────
    ax = axes[0]
    ax.plot(res.index, res['price'],     color='#58a6ff', lw=1,   label='Price',         alpha=0.8)
    ax.plot(res.index, res['EMA_Short'], color='#ffa657', lw=1.5, label=f'EMA {bt.ema_short}')
    ax.plot(res.index, res['EMA_Long'],  color='#ff7b72', lw=1.5, label=f'EMA {bt.ema_long}')

    # shade crossover regions
    ax.fill_between(res.index,
                    res['EMA_Short'], res['EMA_Long'],
                    where=res['EMA_Short'] > res['EMA_Long'],
                    alpha=0.15, color='#3fb950', label='Bullish zone')
    ax.fill_between(res.index,
                    res['EMA_Short'], res['EMA_Long'],
                    where=res['EMA_Short'] < res['EMA_Long'],
                    alpha=0.15, color='#ff7b72', label='Bearish zone')

    ax.set_ylabel('Price', fontsize=10)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

    # ── Panel 2: Cumulative returns ────────────────────────────────
    ax2 = axes[1]
    ax2.plot(res.index, res['cstrategy'], color='#3fb950', lw=2, label='EMA Strategy')
    ax2.plot(res.index, res['creturns'],  color='#8b949e', lw=1.5, ls='--', label='Buy & Hold')
    ax2.axhline(1, color='#30363d', ls=':', lw=1)
    ax2.set_ylabel('Cum. Return', fontsize=10)
    ax2.legend(fontsize=8, loc='upper left')
    ax2.grid(True, alpha=0.3)

    # ── Panel 3: Drawdown ──────────────────────────────────────────
    ax3 = axes[2]
    roll_max  = res['cstrategy'].cummax()
    drawdown  = (res['cstrategy'] - roll_max) / roll_max * 100
    ax3.fill_between(res.index, drawdown, 0, alpha=0.6, color='#da3633')
    ax3.plot(res.index, drawdown, color='#ff7b72', lw=0.8)
    ax3.set_ylabel('Drawdown %', fontsize=10)
    ax3.grid(True, alpha=0.3)

    # ── Panel 4: Position ──────────────────────────────────────────
    ax4 = axes[3]
    ax4.fill_between(res.index, res['position'], 0,
                     where=res['position'] > 0, color='#3fb950', alpha=0.7, label='Long')
    ax4.fill_between(res.index, res['position'], 0,
                     where=res['position'] < 0, color='#da3633', alpha=0.7, label='Short')
    ax4.set_ylabel('Position', fontsize=10)
    ax4.set_ylim(-1.5, 1.5)
    ax4.legend(fontsize=8, loc='upper left')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('backtest_chart.png', dpi=150, bbox_inches='tight',
                facecolor='#0d1117')
    plt.show()
    print('Chart saved → backtest_chart.png')


plot_strategy(bt)

## 9. 🔍 Parameter Optimisation

Grid-search every EMA short/long combination and find the one with the best strategy performance.

In [ ]:
def optimize_ema(backtester: EMABacktester,
                 short_range=(5, 30, 2),
                 long_range=(20, 100, 5)) -> tuple:
    """
    Grid-search best EMA pair.

    Parameters
    ----------
    short_range, long_range : (start, stop, step)
    """
    combos  = list(product(range(*short_range), range(*long_range)))
    results = []

    for s, l in combos:
        if s >= l:                   # skip invalid combos
            results.append(-np.inf)
            continue
        backtester.set_parameters(s, l)
        perf, _ = backtester.test_strategy()
        results.append(perf)

    best_idx   = int(np.argmax(results))
    best_perf  = results[best_idx]
    opt_s, opt_l = combos[best_idx]

    # Apply optimal params
    backtester.set_parameters(opt_s, opt_l)
    backtester.test_strategy()

    df_results = pd.DataFrame(combos, columns=['EMA_Short', 'EMA_Long'])
    df_results['performance'] = results
    df_results = df_results[df_results['performance'] > -np.inf]
    backtester.results_overview = df_results

    return (opt_s, opt_l), round(best_perf, 6)


print('Running optimisation (this may take ~30 seconds)…')
opt_params, opt_perf = optimize_ema(bt)

print(f'\n✅ Best EMA pair    : Short={opt_params[0]}, Long={opt_params[1]}')
print(f'   Best performance : {opt_perf:.4f}x  ({(opt_perf-1)*100:.2f}%)')

print('\nTop 10 EMA combinations:')
bt.results_overview.sort_values('performance', ascending=False).head(10)

In [ ]:
# ── Heatmap of performance across all EMA combinations ─────────────
pivot = bt.results_overview.pivot_table(
    index='EMA_Short', columns='EMA_Long', values='performance'
)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
               vmin=pivot.values[np.isfinite(pivot.values)].min(),
               vmax=pivot.values[np.isfinite(pivot.values)].max())

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=7, rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
ax.set_xlabel('EMA Long', fontsize=11)
ax.set_ylabel('EMA Short', fontsize=11)
ax.set_title(f'Optimisation Heatmap — {CONFIG["SYMBOL"]}', fontsize=14, pad=15)

# Mark optimal
opt_row = list(pivot.index).index(opt_params[0])
opt_col = list(pivot.columns).index(opt_params[1])
ax.add_patch(plt.Rectangle((opt_col-0.5, opt_row-0.5), 1, 1,
             fill=False, edgecolor='white', lw=2))
ax.text(opt_col, opt_row, '★', ha='center', va='center',
        fontsize=12, color='white', fontweight='bold')

plt.colorbar(im, ax=ax, label='Strategy Performance (multiplier)')
plt.tight_layout()
plt.savefig('optimisation_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

## 10. 🛡️ Risk Management

Three layers of risk control:
1. **Position sizing** — size lot based on % account risk
2. **SL / TP** — computed in price units from pip distance
3. **Trailing Stop** — locks in profit as trade moves in your favour
4. **Drawdown circuit breaker** — halts bot if losses exceed threshold

In [ ]:
def pip_value(symbol: str) -> float:
    """Pip size: 0.0001 for most FX, 0.01 for JPY, 0.1 for Gold/Silver."""
    info = mt5.symbol_info(symbol)
    if symbol in ('XAUUSD', 'XAGUSD'):
        return 0.1
    if 'JPY' in symbol:
        return 0.01
    if info:
        return info.point * 10
    return 0.0001


def calculate_lot_size(balance: float, risk_pct: float,
                       sl_pips: int, symbol: str) -> float:
    """
    Fixed fractional position sizing.
    risk_amount = balance × (risk_pct / 100)
    lots        = risk_amount / (sl_pips × pip_cost)
    """
    pip      = pip_value(symbol)
    risk_amt = balance * (risk_pct / 100)
    pip_cost = pip * 10        # approx per standard lot
    raw_lots = risk_amt / (sl_pips * pip_cost)

    info     = mt5.symbol_info(symbol)
    min_lot  = info.volume_min  if info else 0.01
    lot_step = info.volume_step if info else 0.01
    lots     = max(round(raw_lots / lot_step) * lot_step, min_lot)
    return round(lots, 2)


def get_sl_tp(order_type: str, price: float, symbol: str,
              sl_pips: int, tp_pips: int) -> tuple:
    """Convert pip distances to price levels."""
    pip    = pip_value(symbol)
    sl_d   = sl_pips * pip
    tp_d   = tp_pips * pip
    if order_type == 'BUY':
        return round(price - sl_d, 5), round(price + tp_d, 5)
    return round(price + sl_d, 5), round(price - tp_d, 5)


def check_drawdown(peak_balance: float, current_equity: float,
                   max_dd_pct: float) -> bool:
    """Returns True when drawdown limit is breached."""
    if peak_balance <= 0:
        return False
    dd = (peak_balance - current_equity) / peak_balance * 100
    return dd >= max_dd_pct


def update_trailing_stop(position, trail_pips: int, symbol: str) -> bool:
    """Move SL to trail price by trail_pips if it improves the SL."""
    pip  = pip_value(symbol)
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        return False
    if position.type == 0:   # BUY
        new_sl = round(tick.bid - trail_pips * pip, 5)
        if new_sl <= position.sl:
            return False
    else:                    # SELL
        new_sl = round(tick.ask + trail_pips * pip, 5)
        if new_sl >= position.sl or position.sl == 0:
            return False
    request = {
        'action':   mt5.TRADE_ACTION_SLTP,
        'symbol':   symbol,
        'sl':       new_sl,
        'tp':       position.tp,
        'position': position.ticket,
    }
    result = mt5.order_send(request)
    return result.retcode == mt5.TRADE_RETCODE_DONE


# ── Quick demo ─────────────────────────────────────────────────────
acct = get_account_info()
bal  = acct.get('balance', 10000)
sym  = CONFIG['SYMBOL']
lot  = calculate_lot_size(bal, CONFIG['RISK_PER_TRADE_PCT'], CONFIG['SL_PIPS'], sym)
tick = mt5.symbol_info_tick(sym)
ask  = tick.ask if tick else 0
sl, tp = get_sl_tp('BUY', ask, sym, CONFIG['SL_PIPS'], CONFIG['TP_PIPS'])

print(f'📐 Risk Sizing Demo for {sym}')
print(f'  Balance:        ${bal:.2f}')
print(f'  Risk per trade: {CONFIG["RISK_PER_TRADE_PCT"]}%  → ${bal * CONFIG["RISK_PER_TRADE_PCT"]/100:.2f}')
print(f'  SL pips:        {CONFIG["SL_PIPS"]} | TP pips: {CONFIG["TP_PIPS"]}')
print(f'  Calculated lots:{lot}')
print(f'  Entry (ask):    {ask:.5f}')
print(f'  Stop Loss:      {sl:.5f}')
print(f'  Take Profit:    {tp:.5f}')

## 11. 📡 Live Signal Generation

Reads the last *closed* candle (index `-2`) to avoid repainting on the open candle.

In [ ]:
def get_signal(symbol: str, ema_short: int, ema_long: int,
               timeframe: int, bars: int = 100) -> str:
    """
    Returns 'BUY', 'SELL', or 'HOLD'.
    Decision is based on the last CLOSED candle (index -2) to avoid repainting.
    """
    df = fetch_ohlcv(symbol, timeframe, bars)
    if df.empty:
        return 'HOLD'
    df['EMA_Short'] = df['price'].ewm(span=ema_short, adjust=False).mean()
    df['EMA_Long']  = df['price'].ewm(span=ema_long,  adjust=False).mean()

    prev_s = df['EMA_Short'].iloc[-3]
    prev_l = df['EMA_Long'].iloc[-3]
    curr_s = df['EMA_Short'].iloc[-2]
    curr_l = df['EMA_Long'].iloc[-2]

    if prev_s <= prev_l and curr_s > curr_l:
        return 'BUY'
    elif prev_s >= prev_l and curr_s < curr_l:
        return 'SELL'
    return 'HOLD'


# ── Check current signal ────────────────────────────────────────────
signal = get_signal(CONFIG['SYMBOL'], CONFIG['EMA_SHORT'],
                    CONFIG['EMA_LONG'],  CONFIG['TIMEFRAME'])

emoji = {'BUY': '🟢', 'SELL': '🔴', 'HOLD': '⚪'}[signal]
print(f'{emoji} Current signal for {CONFIG["SYMBOL"]}: {signal}')
print(f'   EMA {CONFIG["EMA_SHORT"]}/{CONFIG["EMA_LONG"]}  |  {CONFIG["TIMEFRAME"]} timeframe')

## 12. 📤 Order Execution

In [ ]:
def get_open_position(symbol: str, magic: int):
    """Return the bot's open position for this symbol, or None."""
    positions = mt5.positions_get(symbol=symbol)
    if positions is None:
        return None
    for p in positions:
        if p.magic == magic:
            return p
    return None


def place_order(signal: str, symbol: str, lot: float,
                sl: float, tp: float, magic: int,
                comment: str = 'EMABot') -> dict:
    """Send a market order to MT5."""
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        return {'success': False, 'error': 'No tick data'}

    order_type = mt5.ORDER_TYPE_BUY  if signal == 'BUY'  else mt5.ORDER_TYPE_SELL
    price      = tick.ask            if signal == 'BUY'  else tick.bid

    request = {
        'action':       mt5.TRADE_ACTION_DEAL,
        'symbol':       symbol,
        'volume':       lot,
        'type':         order_type,
        'price':        price,
        'sl':           sl,
        'tp':           tp,
        'deviation':    20,
        'magic':        magic,
        'comment':      comment,
        'type_time':    mt5.ORDER_TIME_GTC,
        'type_filling': mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        return {'success': False, 'error': result.comment,
                'retcode': result.retcode}
    return {'success': True, 'ticket': result.order,
            'price': price, 'lot': lot}


def close_position(position) -> dict:
    """Close an existing MT5 position by ticket."""
    tick = mt5.symbol_info_tick(position.symbol)
    if tick is None:
        return {'success': False, 'error': 'No tick'}

    close_type  = (mt5.ORDER_TYPE_SELL if position.type == 0
                   else mt5.ORDER_TYPE_BUY)
    close_price = tick.bid if position.type == 0 else tick.ask

    request = {
        'action':       mt5.TRADE_ACTION_DEAL,
        'symbol':       position.symbol,
        'volume':       position.volume,
        'type':         close_type,
        'position':     position.ticket,
        'price':        close_price,
        'deviation':    20,
        'magic':        position.magic,
        'comment':      'EMABot-Close',
        'type_time':    mt5.ORDER_TIME_GTC,
        'type_filling': mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        return {'success': False, 'error': result.comment}
    return {'success': True, 'ticket': result.order}


print('✅ Order execution functions defined.')
print('\nℹ️  place_order() and close_position() are called automatically by the bot loop.')
print('   To manually place a test order on demo, uncomment the cell below.')

In [ ]:
# ── Manual test order (DEMO ONLY — uncomment to test) ──────────────

# tick  = mt5.symbol_info_tick(CONFIG['SYMBOL'])
# price = tick.ask
# sl, tp = get_sl_tp('BUY', price, CONFIG['SYMBOL'],
#                    CONFIG['SL_PIPS'], CONFIG['TP_PIPS'])
# lot = calculate_lot_size(
#     get_account_info()['balance'],
#     CONFIG['RISK_PER_TRADE_PCT'],
#     CONFIG['SL_PIPS'],
#     CONFIG['SYMBOL']
# )
# result = place_order('BUY', CONFIG['SYMBOL'], lot, sl, tp, CONFIG['MAGIC'])
# print(result)

## 13. 📱 Telegram Integration

**Setup:**
1. Message `@BotFather` on Telegram → `/newbot` → copy the **token**
2. Message `@userinfobot` → copy your **chat ID**
3. Paste both into `CONFIG` above

**Available commands:**

| Command | Action |
|---|---|
| `/start` | Opens inline keyboard dashboard |
| `/status` | Bot state, trades, PnL |
| `/balance` | Full account breakdown |
| `/signal` | Current EMA signal |
| `/trades` | Open positions |
| `/backtest` | Run backtest + metrics |
| `/optimize` | Grid-search best EMA combo |
| `/settings` | View current CONFIG |

In [ ]:
# ── Global bot state ───────────────────────────────────────────────
bot_state = {
    'running':      False,
    'trades_today': 0,
    'total_pnl':    0.0,
    'peak_balance': 0.0,
    'tg_app':       None,
}


# ── Fire-and-forget notification ───────────────────────────────────
async def send_telegram(text: str):
    try:
        app = bot_state.get('tg_app')
        if app:
            await app.bot.send_message(
                chat_id=CONFIG['TG_CHAT_ID'],
                text=text,
                parse_mode='Markdown'
            )
    except Exception as e:
        log.error(f'Telegram error: {e}')


# ── Command handlers ───────────────────────────────────────────────
async def cmd_start(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    kb = [
        [InlineKeyboardButton('▶️ Start Bot',   callback_data='start_bot'),
         InlineKeyboardButton('⏹ Stop Bot',    callback_data='stop_bot')],
        [InlineKeyboardButton('📊 Status',      callback_data='status'),
         InlineKeyboardButton('💰 Balance',     callback_data='balance')],
        [InlineKeyboardButton('📈 Backtest',    callback_data='backtest'),
         InlineKeyboardButton('🔍 Signal',      callback_data='signal')],
        [InlineKeyboardButton('📋 Open Trades', callback_data='trades'),
         InlineKeyboardButton('⚙️ Settings',    callback_data='settings')],
    ]
    await update.message.reply_text(
        '🤖 *EMA CrossOver Bot*\nChoose an action:',
        reply_markup=InlineKeyboardMarkup(kb),
        parse_mode='Markdown'
    )


async def cmd_status(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    acct   = get_account_info()
    status = '🟢 Running' if bot_state['running'] else '🔴 Stopped'
    msg = (f'*BOT STATUS*\n'
           f'State: {status}\n'
           f'Symbol: `{CONFIG["SYMBOL"]}`\n'
           f'EMA: `{CONFIG["EMA_SHORT"]}/{CONFIG["EMA_LONG"]}`\n'
           f'Balance: `${acct.get("balance", 0):.2f}`\n'
           f'Equity:  `${acct.get("equity",  0):.2f}`\n'
           f'Trades today: `{bot_state["trades_today"]}`\n'
           f'Total PnL: `${bot_state["total_pnl"]:.2f}`')
    await update.message.reply_text(msg, parse_mode='Markdown')


async def cmd_balance(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    acct = get_account_info()
    msg = (f'💰 *ACCOUNT*\n'
           f'Balance:      `${acct.get("balance",      0):.2f}`\n'
           f'Equity:       `${acct.get("equity",       0):.2f}`\n'
           f'Free Margin:  `${acct.get("free_margin",  0):.2f}`\n'
           f'Open P&L:     `${acct.get("profit",       0):.2f}`\n'
           f'Margin Level: `{acct.get("margin_level",  0):.1f}%`')
    await update.message.reply_text(msg, parse_mode='Markdown')


async def cmd_signal(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    sig   = get_signal(CONFIG['SYMBOL'], CONFIG['EMA_SHORT'],
                       CONFIG['EMA_LONG'], CONFIG['TIMEFRAME'])
    emoji = '🟢' if sig == 'BUY' else ('🔴' if sig == 'SELL' else '⚪')
    await update.message.reply_text(
        f'{emoji} *Current Signal*: `{sig}`\n'
        f'Symbol: `{CONFIG["SYMBOL"]}` | '
        f'EMA {CONFIG["EMA_SHORT"]}/{CONFIG["EMA_LONG"]}',
        parse_mode='Markdown'
    )


async def cmd_trades(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    pos = get_open_position(CONFIG['SYMBOL'], CONFIG['MAGIC'])
    if pos is None:
        await update.message.reply_text('📋 No open positions.')
        return
    direction = 'BUY' if pos.type == 0 else 'SELL'
    msg = (f'📋 *OPEN POSITION*\n'
           f'Ticket:      `{pos.ticket}`\n'
           f'Direction:   `{direction}`\n'
           f'Volume:      `{pos.volume}`\n'
           f'Open Price:  `{pos.price_open:.5f}`\n'
           f'SL: `{pos.sl:.5f}` | TP: `{pos.tp:.5f}`\n'
           f'Current P&L: `${pos.profit:.2f}`')
    await update.message.reply_text(msg, parse_mode='Markdown')


async def cmd_backtest(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text('⏳ Running backtest…')
    try:
        b = EMABacktester(CONFIG['SYMBOL'], CONFIG['EMA_SHORT'], CONFIG['EMA_LONG'])
        b.test_strategy()
        m = compute_metrics(b.results['strategy'], b.results['cstrategy'])
        lines = [f'📊 *BACKTEST — {CONFIG["SYMBOL"]}*',
                 f'EMA {CONFIG["EMA_SHORT"]}/{CONFIG["EMA_LONG"]}', '─'*30]
        for k, v in m.items():
            lines.append(f'  {k}: `{v}`')
        await update.message.reply_text('\n'.join(lines), parse_mode='Markdown')
    except Exception as e:
        await update.message.reply_text(f'❌ Error: {e}')


async def cmd_optimize(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text('⏳ Optimising EMA parameters…')
    try:
        b = EMABacktester(CONFIG['SYMBOL'], CONFIG['EMA_SHORT'], CONFIG['EMA_LONG'])
        opt, perf = optimize_ema(b)
        CONFIG['EMA_SHORT'] = opt[0]
        CONFIG['EMA_LONG']  = opt[1]
        msg = (f'✅ *OPTIMISATION COMPLETE*\n'
               f'Best EMA: Short=`{opt[0]}`, Long=`{opt[1]}`\n'
               f'Performance: `{perf:.4f}x`\nConfig updated.')
        await update.message.reply_text(msg, parse_mode='Markdown')
    except Exception as e:
        await update.message.reply_text(f'❌ Error: {e}')


async def cmd_settings(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    msg = (f'⚙️ *CURRENT SETTINGS*\n'
           f'Symbol:      `{CONFIG["SYMBOL"]}`\n'
           f'EMA Short:   `{CONFIG["EMA_SHORT"]}`\n'
           f'EMA Long:    `{CONFIG["EMA_LONG"]}`\n'
           f'SL (pips):   `{CONFIG["SL_PIPS"]}`\n'
           f'TP (pips):   `{CONFIG["TP_PIPS"]}`\n'
           f'Lot Size:    `{CONFIG["LOT_SIZE"]}`\n'
           f'Risk %:      `{CONFIG["RISK_PER_TRADE_PCT"]}%`\n'
           f'Max DD %:    `{CONFIG["MAX_DRAWDOWN_PCT"]}%`\n'
           f'Trailing SL: `{CONFIG["TRAILING_SL"]}`')
    await update.message.reply_text(msg, parse_mode='Markdown')


# ── Button handler ─────────────────────────────────────────────────
async def button_handler(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    data = query.data
    update.message = query.message

    if data == 'start_bot':
        bot_state['running'] = True
        await query.message.reply_text('✅ Bot *started*.', parse_mode='Markdown')
    elif data == 'stop_bot':
        bot_state['running'] = False
        await query.message.reply_text('⏹ Bot *stopped*.', parse_mode='Markdown')
    elif data == 'status':   await cmd_status(update, ctx)
    elif data == 'balance':  await cmd_balance(update, ctx)
    elif data == 'backtest': await cmd_backtest(update, ctx)
    elif data == 'signal':   await cmd_signal(update, ctx)
    elif data == 'trades':   await cmd_trades(update, ctx)
    elif data == 'settings': await cmd_settings(update, ctx)


print('✅ All Telegram handlers defined.')

## 14. ⏱️ Bot Loop & Scheduler

In [ ]:
def bot_tick():
    """One iteration of the main trading loop."""
    if not bot_state['running']:
        return

    symbol = CONFIG['SYMBOL']
    acct   = get_account_info()
    if not acct:
        log.warning('Could not fetch account info.')
        return

    equity  = acct['equity']
    balance = acct['balance']

    # ── Track peak for drawdown ────────────────────────────────────
    if equity > bot_state['peak_balance']:
        bot_state['peak_balance'] = equity

    # ── Drawdown circuit breaker ───────────────────────────────────
    if check_drawdown(bot_state['peak_balance'], equity, CONFIG['MAX_DRAWDOWN_PCT']):
        msg = (f'🚨 *MAX DRAWDOWN REACHED*\n'
               f'Peak: ${bot_state["peak_balance"]:.2f} | Equity: ${equity:.2f}\nBot halted.')
        log.error(msg)
        bot_state['running'] = False
        asyncio.run(send_telegram(msg))
        return

    # ── Trailing stop update ───────────────────────────────────────
    if CONFIG['TRAILING_SL']:
        pos = get_open_position(symbol, CONFIG['MAGIC'])
        if pos:
            update_trailing_stop(pos, CONFIG['TRAIL_PIPS'], symbol)

    # ── Get signal ─────────────────────────────────────────────────
    signal = get_signal(symbol, CONFIG['EMA_SHORT'],
                        CONFIG['EMA_LONG'],  CONFIG['TIMEFRAME'])
    log.info(f'Tick | Signal: {signal} | Equity: ${equity:.2f}')

    if signal == 'HOLD':
        return

    pos = get_open_position(symbol, CONFIG['MAGIC'])

    # ── Opposite signal: close existing position ───────────────────
    if pos:
        pos_is_buy = pos.type == 0
        sig_is_buy = signal == 'BUY'
        if pos_is_buy != sig_is_buy:
            res = close_position(pos)
            if res['success']:
                bot_state['total_pnl'] += pos.profit
                asyncio.run(send_telegram(
                    f'🔄 *POSITION CLOSED*\nSymbol: `{symbol}`\n'
                    f'P&L: `${pos.profit:.2f}`\nReason: {signal} signal'
                ))
        else:
            return   # same direction, keep riding

    # ── Open new position ──────────────────────────────────────────
    tick  = mt5.symbol_info_tick(symbol)
    price = tick.ask if signal == 'BUY' else tick.bid
    sl, tp = get_sl_tp(signal, price, symbol,
                       CONFIG['SL_PIPS'], CONFIG['TP_PIPS'])
    lot  = calculate_lot_size(balance, CONFIG['RISK_PER_TRADE_PCT'],
                              CONFIG['SL_PIPS'], symbol)
    res  = place_order(signal, symbol, lot, sl, tp, CONFIG['MAGIC'])

    if res['success']:
        bot_state['trades_today'] += 1
        emoji = '🟢' if signal == 'BUY' else '🔴'
        asyncio.run(send_telegram(
            f'{emoji} *NEW {signal}*\nSymbol: `{symbol}`\n'
            f'Entry: `{res["price"]:.5f}` | Lots: `{lot}`\n'
            f'SL: `{sl:.5f}` | TP: `{tp:.5f}`\n'
            f'Ticket: `{res["ticket"]}`'
        ))
    else:
        log.error(f'Order failed: {res}')


def run_scheduler():
    """Runs in a background thread — calls bot_tick every N seconds."""
    interval = CONFIG['CHECK_INTERVAL_SEC']
    schedule.every(interval).seconds.do(bot_tick)
    log.info(f'Scheduler started — every {interval}s')
    while True:
        schedule.run_pending()
        time.sleep(1)


print('✅ Bot tick loop and scheduler defined.')

## 15. 🚀 Launch the Bot

> ✅ Make sure you have filled in `CONFIG` in Section 2 before running this cell.

This cell:
1. Connects to MT5
2. Starts the Telegram bot (polling mode)
3. Starts the scheduler in a background thread
4. Sends a startup notification to your Telegram

**To stop:** interrupt the kernel (`■` button) or send `/stop_bot` in Telegram.

In [ ]:
def launch_bot():
    print('═' * 55)
    print('   EMA CROSSOVER BOT  —  Starting up')
    print('═' * 55)

    # ── Connect MT5 ─────────────────────────────────────────────
    if not connect_mt5():
        print('❌ MT5 connection failed. Check credentials in CONFIG.')
        return

    acct = get_account_info()
    bot_state['peak_balance'] = acct.get('balance', 0)
    print(f'✅ MT5 connected | Balance: ${acct.get("balance",0):.2f} {acct.get("currency","")}')

    # ── Build Telegram app ───────────────────────────────────────
    tg_app = ApplicationBuilder().token(CONFIG['TG_TOKEN']).build()

    tg_app.add_handler(CommandHandler('start',    cmd_start))
    tg_app.add_handler(CommandHandler('status',   cmd_status))
    tg_app.add_handler(CommandHandler('balance',  cmd_balance))
    tg_app.add_handler(CommandHandler('signal',   cmd_signal))
    tg_app.add_handler(CommandHandler('trades',   cmd_trades))
    tg_app.add_handler(CommandHandler('backtest', cmd_backtest))
    tg_app.add_handler(CommandHandler('optimize', cmd_optimize))
    tg_app.add_handler(CommandHandler('settings', cmd_settings))
    tg_app.add_handler(CallbackQueryHandler(button_handler))

    bot_state['tg_app'] = tg_app
    print('✅ Telegram handlers registered.')

    # ── Scheduler thread ─────────────────────────────────────────
    sched_thread = threading.Thread(target=run_scheduler, daemon=True)
    sched_thread.start()
    print(f'✅ Scheduler started (every {CONFIG["CHECK_INTERVAL_SEC"]}s).')

    # ── Startup Telegram message ─────────────────────────────────
    async def on_startup(app):
        await send_telegram(
            f'🚀 *EMA Bot Online*\n'
            f'Symbol: `{CONFIG["SYMBOL"]}` | EMA {CONFIG["EMA_SHORT"]}/{CONFIG["EMA_LONG"]}\n'
            f'Balance: `${acct.get("balance",0):.2f}`\n'
            f'Use /start for the control panel.'
        )

    tg_app.post_init = on_startup

    print('\n📱 Telegram polling started. Open your bot and type /start')
    print('   Press ■ (Stop) in Jupyter to shut down.\n')

    try:
        tg_app.run_polling()
    finally:
        disconnect_mt5()
        print('Bot stopped. MT5 disconnected.')


launch_bot()

---
## 📚 Reference

### Timeframe constants
| Constant | Description |
|---|---|
| `mt5.TIMEFRAME_M1` | 1 minute |
| `mt5.TIMEFRAME_M15` | 15 minutes |
| `mt5.TIMEFRAME_M30` | 30 minutes |
| `mt5.TIMEFRAME_H1` | 1 hour |
| `mt5.TIMEFRAME_H4` | 4 hours |
| `mt5.TIMEFRAME_D1` | Daily |

### Recommended EMA pairs by instrument
| Instrument | Short EMA | Long EMA | Timeframe |
|---|---|---|---|
| XAUUSD (Gold) | 9 | 21 | H1 |
| EURUSD | 12 | 26 | H4 |
| NAS100 | 9 | 50 | H1 |
| USDJPY | 8 | 21 | H1 |

### Key risk rules
- Never risk more than **1-2% per trade**
- Set `MAX_DRAWDOWN_PCT` to **5-10%** max before the bot halts
- Always run on **demo first** for at least 2 weeks before going live
- Run `/optimize` periodically (monthly) to keep EMA params fresh